# extended covariance forecasting

The extension adds a delayed-feedback mixture of covariance shapes to the audited graph-HAR replication. It addresses a question raised by the first experiment: can a model improve covariance forecast accuracy while failing to reduce portfolio risk?

[Full proofs](docs/theory.md) · [Frozen experimental design](docs/extension-design.json) · [Original replication](Covariance_Replication.ipynb)

![Adaptive covariance architecture](docs/adaptive-architecture.png)

The method was designed after examining the July 2022–December 2024 results. We freeze the three experts, common-trace rule and learning rate before examining the 2025–October 2026 confirmation outputs. The earlier period is development evidence. This is a project extension using established spectral and online-learning ideas, with new computations on this public panel. Mathematical priority is unassessed.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from threadpoolctl import threadpool_limits

from market_data import load_market
from models import BASELINE, GRAPH_MODEL, Forecasts, forecast_grid
from evaluation import bootstrap_means, covariance_losses, forecast_table, point_ratios, portfolio_table
from extension import (
    ADAPTIVE_MODEL, UNIFORM_MODEL, EXPERTS, adaptive_shapes,
    line_graph_spectrum, loss_risk_counterexample,
)

design = json.loads(Path("docs/extension-design.json").read_text())
assert design["experts"] == list(EXPERTS)
assert design["learning_rate"] == .5
display(pd.Series(design))

method                             exponential-weights mixture of unit-trace cova...
experts                                          [HAR-DRD, GHAR(GL,L), HAR-Cholesky]
trace_forecast                                                               HAR-DRD
learning_rate                                                                    0.5
horizons_in_retained_sessions                                             [1, 5, 22]
feedback                           target is scored only after its last retained ...
development_period                                          [2022-07-01, 2024-12-31]
confirmation_period                                         [2025-01-01, 2026-10-23]
primary_confirmation_comparison    adaptive versus uniform mixture under normaliz...
additional_comparisons             [matrix losses, MCS, one-session GMVP and GMVP...
qualification                      method selected after examining the developmen...
primary_confirmation_horizon                                     

## 1. why smaller covariance errors can give worse portfolios

For true covariance $H=\operatorname{diag}(1,4)$, compare forecasts $A=2H$ and $C=\operatorname{diag}(1,3)$. $C$ is closer under all three matrix losses, but its GMVP has true variance $0.8125$, versus $0.8$ from $A$. Both weight vectors are long-only.

The example proves that the portfolio claim requires its own test. A common positive scaling of a covariance forecast changes matrix losses while leaving GMVP weights unchanged. The extension therefore learns **unit-trace shapes** and fixes total variance to the HAR-DRD trace. The notebook includes each raw expert, each nonbaseline expert under that common trace, a uniform mixture and the adaptive mixture.

In [2]:
display(loss_risk_counterexample().round(6))

,Euclidean,Frobenius,excess QLIKE,first weight,true portfolio variance
forecast,,,,,
scaled truth,4.123106,4.123106,0.386294,0.80,0.8000
closer forecast,1.000000,1.000000,0.045651,0.75,0.8125


## 2. what the line graph does

With $B$ the stock–pair incidence matrix and $n=27$ stocks,

$$W_L=\frac{B^\top B-2I}{2(n-2)}.$$

Its nonconstant eigenvalues have absolute value at most $0.46$. Therefore

$$\|W_L^kX-JX\|_F\le0.46^k\|X-JX\|_F.$$

This is an exact property of repeated linear aggregation. It gives a mechanism for smoothing pair contrasts, without identifying an optimal empirical forecasting depth.

In [3]:
spectrum = line_graph_spectrum(27)
display(spectrum)
contraction = pd.DataFrame({
    "graph passes": np.arange(7),
    "upper bound on centered norm": .46 ** np.arange(7),
})
fig = px.line(
    contraction, x="graph passes", y="upper bound on centered norm", markers=True,
    title="27-stock line graph · centered-feature contraction bound",
)
fig.update_layout(template="plotly_white", height=350)
fig.show(renderer="plotly_mimetype")

,eigenvalue,multiplicity
0,1.00,1
1,0.46,26
2,-0.04,324


## 3. delayed adaptive covariance shapes

At origin $t$, normalize each expert forecast to $C_{t,j}=\hat\Sigma_{t,j}/\operatorname{tr}(\hat\Sigma_{t,j})$. Use only previously resolved targets to update

$$p_{t,j}\propto\exp\left[-0.5\sum_{s\in O_t}\frac12\|C_{s,j}-C_s^*\|_F^2\right],$$

then predict

$$\hat\Sigma_t^{\mathrm{mix}}=
\operatorname{tr}(\hat\Sigma_t^{\mathrm{HAR-DRD}})
\sum_jp_{t,j}C_{t,j}.$$

The convex mixture is positive definite and retains the chosen total variance. The loss on trace-one positive-semidefinite shapes lies in $[0,1]$. For one-session forecasts, the proof gives cumulative regret at most $2\log3$ against the best fixed shape expert from the same run. At longer horizons, unresolved labels add the explicit delay term in theorem 5. These bounds concern shape loss, while raw losses and portfolio outcomes are measured separately.

The three experts and learning rate are held fixed across all horizons. A target starting at retained position $s$ and spanning $h$ sessions is used only at a later origin strictly after $s+h-1$. In particular, future targets cannot affect earlier weights.

In [4]:
with threadpool_limits(limits=1):
    panel = load_market(
        "syncster", Path("data/syncster"), "2016-01-04", design["confirmation_period"][1],
    )
    base_forecasts = forecast_grid(
        panel, horizons=tuple(design["horizons_in_retained_sessions"]), lookback=1000,
    )
    mixtures = {
        h: adaptive_shapes(result, h, learning_rate=design["learning_rate"])
        for h, result in base_forecasts.items()
    }
print(f"{len(panel.symbols)} stocks · {len(panel.dates):,} retained sessions")
print(f"all forecasts begin {base_forecasts[1].dates[0]:%Y-%m-%d}")
display(pd.DataFrame({h: result.regret for h, result in mixtures.items()}).T.round(6))

27 stocks · 2,090 retained sessions
all forecasts begin 2022-07-01


,horizon,origins,learning rate,adaptive cumulative shape loss,uniform cumulative shape loss,best fixed shape loss,adaptive regret,regret upper bound,unresolved-label count,feedback-delay term,general delayed bound
1,1.0,1066.0,0.5,24.608478,24.636328,24.688054,-0.079576,2.197225,0.0,0.00,68.822225
5,5.0,1062.0,0.5,12.251741,12.269225,12.284143,-0.032403,598.322225,4238.0,529.75,598.322225
22,22.0,1045.0,0.5,10.369571,10.383618,10.233738,0.135833,2781.759725,21714.0,2714.25,2781.759725


## 4. development and confirmation

The primary confirmation comparison is adaptive versus uniform one-session **shape loss**. Five- and 22-session comparisons are secondary. Confidence intervals use a paired stationary bootstrap with 10,000 draws and mean block length $\max(5,h)$, fixed before running the extension.

Development targets ending in 2025 are omitted from the development table. Confirmation forecasts start in 2025, with mixture weights carried forward from earlier, available outcomes. The full-run regret bound uses the initial uniform prior in 2022. It is not reset and asserted again at the confirmation boundary.

The public calendar and data-source limitations from the replication continue to apply. All observations, targets and weights in a comparison refer to the same retained sessions.

In [5]:
cutoff = pd.Timestamp(design["confirmation_period"][0])
period_results = {}
shape_records = []

for h, extension in mixtures.items():
    result = extension.forecasts
    final_target_date = panel.dates[result.positions + h - 1]
    masks = {
        "development": (result.dates < cutoff) & (final_target_date < cutoff),
        "confirmation": result.dates >= cutoff,
    }
    for period, mask in masks.items():
        if not mask.any():
            raise ValueError(f"No {period} origins for horizon {h}.")
        selected = Forecasts(
            result.dates[mask], result.positions[mask], result.actual[mask],
            {model: values[mask] for model, values in result.predictions.items()},
            result.audit.copy(),
        )
        period_results[(period, h)] = selected
        adaptive_loss = extension.mixture_loss.to_numpy()[mask]
        uniform_loss = extension.uniform_loss.to_numpy()[mask]
        difference = adaptive_loss - uniform_loss
        samples = bootstrap_means(
            difference[:, None], 10_000, max(5, h), np.random.default_rng(2026 + h),
        )[:, 0]
        shape_records.append({
            "period": period, "sessions": h, "origins": len(adaptive_loss),
            "adaptive mean shape loss": adaptive_loss.mean(),
            "uniform mean shape loss": uniform_loss.mean(),
            "adaptive minus uniform": difference.mean(),
            "95% lower": np.quantile(samples, .025),
            "95% upper": np.quantile(samples, .975),
        })

shape_results = pd.DataFrame(shape_records).set_index(["period", "sessions"])
display(shape_results.round(8))

,,origins,adaptive mean shape loss,uniform mean shape loss,adaptive minus uniform,95% lower,95% upper
period,sessions,,,,,,
development,1,623,0.021901,0.021923,-2.191000e-05,-0.000032,-0.000012
confirmation,1,443,0.024749,0.024781,-3.206000e-05,-0.000070,0.000004
development,5,619,0.010688,0.010708,-1.963000e-05,-0.000025,-0.000014
confirmation,5,439,0.012699,0.012712,-1.273000e-05,-0.000032,0.000009
development,22,602,0.008673,0.008696,-2.307000e-05,-0.000035,-0.000013
confirmation,22,422,0.011994,0.011994,-2.300000e-07,-0.000048,0.000076


In [6]:
loss_results = {}
tables = {}
with threadpool_limits(limits=1):
    for key, result in period_results.items():
        rows = covariance_losses(result)
        loss_results[key] = rows
        tables[key] = forecast_table(rows, draws=10_000, block=max(2, key[1]))
for period in ("development", "confirmation"):
    for h in (1, 5, 22):
        print(f"{period} · {h} sessions · raw covariance losses relative to HAR-DRD")
        display(tables[(period, h)].round(4))

development · 1 sessions · raw covariance losses relative to HAR-DRD


Euclidean              Frobenius               \
                                ratio rank   MCS p     ratio rank   MCS p   
model                                                                       
HAR-DRD                        1.0000    6  0.0005    1.0000    6  0.0003   
GHAR(GL,L)                     0.9835    2  0.0010    0.9855    2  0.0003   
HAR-Cholesky                   0.9376    1  1.0000    0.9388    1  1.0000   
GHAR(GL,L) · common trace      0.9865    3  0.0005    0.9880    3  0.0003   
HAR-Cholesky · common trace    1.0107    7  0.0001    1.0190    7  0.0001   
Uniform shape                  0.9890    5  0.0005    0.9921    5  0.0003   
Adaptive shape                 0.9886    4  0.0005    0.9916    4  0.0003   

                              QLIKE               
                              ratio rank   MCS p  
model                                             
HAR-DRD                      1.0000    5  0.0001  
GHAR(GL,L)                   0.9851    1  1.0000  
HAR-Cholesky                 1.0790    7  0.0001  
GHAR(GL,L) · common trace    0.9865    2  0.0249  
HAR-Cholesky · common trace  1.0055    6  0.0001  
Uniform shape                0.9893    4  0.0001  
Adaptive shape               0.9889    3  0.0030

development · 5 sessions · raw covariance losses relative to HAR-DRD


Euclidean              Frobenius               \
                                ratio rank   MCS p     ratio rank   MCS p   
model                                                                       
HAR-DRD                        1.0000    7  0.0001    1.0000    7  0.0001   
GHAR(GL,L)                     0.9740    5  0.0001    0.9781    5  0.0001   
HAR-Cholesky                   0.8741    1  1.0000    0.8787    1  1.0000   
GHAR(GL,L) · common trace      0.9771    6  0.0001    0.9807    6  0.0001   
HAR-Cholesky · common trace    0.9679    2  0.0001    0.9763    4  0.0001   
Uniform shape                  0.9721    4  0.0001    0.9754    3  0.0001   
Adaptive shape                 0.9713    3  0.0001    0.9747    2  0.0001   

                              QLIKE               
                              ratio rank   MCS p  
model                                             
HAR-DRD                      1.0000    7  0.0001  
GHAR(GL,L)                   0.9953    1  1.0000  
HAR-Cholesky                 0.9999    6  0.0013  
GHAR(GL,L) · common trace    0.9957    2  0.0486  
HAR-Cholesky · common trace  0.9971    5  0.0013  
Uniform shape                0.9962    4  0.0013  
Adaptive shape               0.9961    3  0.0309

development · 22 sessions · raw covariance losses relative to HAR-DRD


Euclidean              Frobenius               \
                                ratio rank   MCS p     ratio rank   MCS p   
model                                                                       
HAR-DRD                        1.0000    7  0.0001    1.0000    6  0.0001   
GHAR(GL,L)                     0.9999    6  0.0001    1.0045    7  0.0001   
HAR-Cholesky                   0.8086    1  1.0000    0.8143    1  1.0000   
GHAR(GL,L) · common trace      0.9934    5  0.0001    0.9978    5  0.0001   
HAR-Cholesky · common trace    0.9614    2  0.0001    0.9675    2  0.0001   
Uniform shape                  0.9781    4  0.0001    0.9813    4  0.0001   
Adaptive shape                 0.9771    3  0.0001    0.9805    3  0.0001   

                              QLIKE               
                              ratio rank   MCS p  
model                                             
HAR-DRD                      1.0000    7  0.0001  
GHAR(GL,L)                   0.9989    6  0.0066  
HAR-Cholesky                 0.9956    1  1.0000  
GHAR(GL,L) · common trace    0.9988    5  0.0066  
HAR-Cholesky · common trace  0.9985    2  0.0555  
Uniform shape                0.9986    4  0.0066  
Adaptive shape               0.9986    3  0.0555

... (2 outputs left out)

confirmation · 5 sessions · raw covariance losses relative to HAR-DRD


Euclidean              Frobenius               \
                                ratio rank   MCS p     ratio rank   MCS p   
model                                                                       
HAR-DRD                        1.0000    4  0.4269    1.0000    4  0.3299   
GHAR(GL,L)                     1.0068    7  0.4313    1.0074    6  0.3299   
HAR-Cholesky                   0.9702    1  1.0000    0.9666    1  1.0000   
GHAR(GL,L) · common trace      1.0029    5  0.3375    1.0024    5  0.3299   
HAR-Cholesky · common trace    1.0035    6  0.4313    1.0111    7  0.3299   
Uniform shape                  0.9962    2  0.4313    0.9988    2  0.3299   
Adaptive shape                 0.9964    3  0.4313    0.9993    3  0.3299   

                              QLIKE               
                              ratio rank   MCS p  
model                                             
HAR-DRD                      1.0000    6  0.0001  
GHAR(GL,L)                   0.9968    5  0.0071  
HAR-Cholesky                 1.0043    7  0.0002  
GHAR(GL,L) · common trace    0.9967    4  0.0071  
HAR-Cholesky · common trace  0.9962    2  0.7196  
Uniform shape                0.9962    3  0.0071  
Adaptive shape               0.9960    1  1.0000

confirmation · 22 sessions · raw covariance losses relative to HAR-DRD


Euclidean              Frobenius               \
                                ratio rank   MCS p     ratio rank   MCS p   
model                                                                       
HAR-DRD                        1.0000    6  0.2678    1.0000    5  0.3394   
GHAR(GL,L)                     1.0042    7  0.2678    1.0056    6  0.3281   
HAR-Cholesky                   0.9433    1  1.0000    0.9455    1  1.0000   
GHAR(GL,L) · common trace      0.9943    3  0.4158    0.9962    2  0.3485   
HAR-Cholesky · common trace    0.9992    5  0.4158    1.0104    7  0.3485   
Uniform shape                  0.9941    2  0.4158    0.9987    3  0.3485   
Adaptive shape                 0.9945    4  0.4158    0.9997    4  0.3485   

                              QLIKE               
                              ratio rank   MCS p  
model                                             
HAR-DRD                      1.0000    7  0.0049  
GHAR(GL,L)                   0.9993    6  0.0983  
HAR-Cholesky                 0.9993    5  0.1029  
GHAR(GL,L) · common trace    0.9989    4  0.0983  
HAR-Cholesky · common trace  0.9974    1  1.0000  
Uniform shape                0.9983    3  0.1029  
Adaptive shape               0.9982    2  0.1277

## 5. portfolio outcomes

Use the same one-session returns and constraints for every model. Matrix scaling leaves fully invested GMVP weights unchanged, so the raw and common-trace versions of each expert need only one portfolio calculation. The uniform mixture is the direct control for the incremental value of learning the weights.

Annualized risk and turnover remain empirical outcomes. The trace-one regret theorem does not guarantee lower portfolio risk.

In [7]:
portfolio_results = {}
with threadpool_limits(limits=1):
    for period in ("development", "confirmation"):
        portfolio_results[period] = portfolio_table(
            period_results[(period, 1)], panel.portfolio_return,
            selected=[*EXPERTS, UNIFORM_MODEL, ADAPTIVE_MODEL],
        )
        print(period)
        display(portfolio_results[period].round(6))

weight_plot = mixtures[1].weights.rename_axis("date").reset_index().melt(
    id_vars="date", var_name="expert", value_name="weight",
)
fig = px.line(
    weight_plot, x="date", y="weight", color="expert",
    title="One-session mixture · weights before each realized outcome",
)
fig.add_vline(x=cutoff.value / 1_000_000, line_dash="dot", line_color="#647880")
fig.update_layout(template="plotly_white", height=420, yaxis_range=[0, 1])
fig.show(renderer="plotly_mimetype")

development


,model,portfolio,annual risk (%),turnover
0,1/N,equal,11.172350,0.007710
1,HAR-DRD,GMVP,9.416480,0.671102
2,HAR-DRD,GMVP+,9.332649,0.436802
3,"GHAR(GL,L)",GMVP,9.479839,0.608246
4,"GHAR(GL,L)",GMVP+,9.385162,0.415817
5,HAR-Cholesky,GMVP,9.416595,0.594439
6,HAR-Cholesky,GMVP+,9.317761,0.356980
7,Uniform shape,GMVP,9.378374,0.582869
8,Uniform shape,GMVP+,9.316166,0.386218
9,Adaptive shape,GMVP,9.376297,0.583074


confirmation


,model,portfolio,annual risk (%),turnover
0,1/N,equal,12.596146,0.010125
1,HAR-DRD,GMVP,9.325832,0.457929
2,HAR-DRD,GMVP+,9.435110,0.328546
3,"GHAR(GL,L)",GMVP,30.161618,0.770450
4,"GHAR(GL,L)",GMVP+,9.746767,0.336007
5,HAR-Cholesky,GMVP,9.169764,0.436514
6,HAR-Cholesky,GMVP+,9.404682,0.307518
7,Uniform shape,GMVP,9.221239,0.414509
8,Uniform shape,GMVP+,9.466393,0.304874
9,Adaptive shape,GMVP,9.227863,0.415365


## 6. conditioning and saved results

Inverse-based portfolio weights can be sensitive to nearly singular covariance forecasts. Report eigenvalue conditioning and unrestricted gross leverage as diagnostics; the original forecasts remain unchanged. The exported tables preserve the results used in the interpretation.

In [8]:
table_directory = Path("docs/tables")
table_directory.mkdir(parents=True, exist_ok=True)
shape_results.to_csv(table_directory / "shape-loss.csv")
pd.DataFrame({h: result.regret for h, result in mixtures.items()}).T.to_csv(
    table_directory / "regret.csv", index_label="horizon",
)
for (period, h), table in tables.items():
    table.to_csv(table_directory / f"{period}-matrix-{h}.csv")
for period, table in portfolio_results.items():
    table.to_csv(table_directory / f"{period}-portfolio.csv", index=False)

confirmation = period_results[("confirmation", 1)]
diagnostics = []
for model in [*EXPERTS, UNIFORM_MODEL, ADAPTIVE_MODEL]:
    predicted = confirmation.predictions[model]
    eigenvalues = np.linalg.eigvalsh(predicted)
    condition = eigenvalues[:, -1] / eigenvalues[:, 0]
    ones = np.ones((len(predicted), predicted.shape[-1], 1))
    raw = np.linalg.solve(predicted, ones)[..., 0]
    weights = raw / raw.sum(axis=1, keepdims=True)
    gross = np.abs(weights).sum(axis=1)
    maximum = int(np.argmax(gross))
    diagnostics.append({
        "model": model,
        "median condition number": np.median(condition),
        "maximum condition number": condition.max(),
        "maximum gross leverage": gross[maximum],
        "maximum leverage date": confirmation.dates[maximum],
        "minimum eigenvalue": eigenvalues[:, 0].min(),
    })
conditioning = pd.DataFrame(diagnostics).set_index("model")
conditioning.to_csv(table_directory / "confirmation-conditioning.csv")
display(conditioning.round(6))

ratio_plot = point_ratios(loss_results[("confirmation", 1)]).loc[
    [BASELINE, GRAPH_MODEL, "HAR-Cholesky", UNIFORM_MODEL, ADAPTIVE_MODEL]
].rename_axis("model").reset_index().melt(
    id_vars="model", var_name="loss", value_name="ratio to har-drd",
)
fig = px.bar(
    ratio_plot, x="model", y="ratio to har-drd", color="loss", barmode="group",
    title="confirmation period · one-session covariance loss ratios",
)
fig.add_hline(y=1, line_dash="dot", line_color="#647880")
fig.update_layout(template="plotly_white", height=420)
fig.show(renderer="plotly_mimetype")

,median condition number,maximum condition number,maximum gross leverage,maximum leverage date,minimum eigenvalue
model,,,,,
HAR-DRD,38.004461,157.232774,2.066720,2025-04-14,0.215093
"GHAR(GL,L)",33.121507,127174.142193,37.049040,2025-04-11,0.001719
HAR-Cholesky,41.060015,247.611408,2.544377,2025-04-10,0.158139
Uniform shape,36.415728,218.932550,2.303684,2025-04-14,0.215690
Adaptive shape,35.699946,223.366929,2.316522,2025-04-11,0.213689


## 7. interpretation and supported claims

Across 443 confirmation origins in 2025–October 2026, adaptive one-session Euclidean, Frobenius and QLIKE loss ratios against HAR-DRD are **0.9977, 0.9979 and 0.9918**. These are descriptive improvements of about 0.23%, 0.21% and 0.82%. Euclidean and Frobenius MCS tests retain the baseline too; QLIKE selects the adaptive model from this candidate set.

The primary adaptive-versus-uniform mean shape-loss difference is about **−0.0000321**, with a paired 95% stationary-bootstrap interval of approximately **[−0.000070, 0.000004]**. It crosses zero. This run therefore does not establish a directional confirmation improvement on the primary endpoint. The secondary five- and 22-session shape intervals also cross zero.

Adaptive unrestricted GMVP risk is **9.2279%**, versus 9.3258% for HAR-DRD and 9.2212% for the uniform mixture. Long-only GMVP+ risk is **9.4912%**, versus 9.4351% and 9.4664% respectively. These are annualized gross realized risks; transaction costs are excluded. Portfolio improvement is not established, and learning the mixture does not beat its uniform control on either confirmation risk measure.

The proofs establish exact graph aggregation, a loss–risk counterexample, a conditional relative-spectral risk bound, positive covariance mixing, and finite-sequence shape-loss regret bounds. Regression checks verify the formulas and that changing an unresolved outcome leaves earlier weights unchanged. These conditional mathematical results do not prove the original paper's empirical benchmark or establish mathematical priority.

[Detailed findings and use cases](docs/extension-results.md) · [Saved result tables](docs/tables)